In [16]:

# No terminal, uma vez: python -m pip install xlrd pandas
import pandas as pd



In [32]:
path = r"H:\python\3atapa\Tabela 1.1.1.xlsx"

In [35]:
indicador_1 = pd.read_excel(path, engine="xlsx")

ValueError: Unknown engine: xlsx

In [13]:
indicador_1


,"Tabela 1.1.1 - Número médio de horas semanais dedicadas aos cuidados de pessoas e/ou afazeres domésticos das pessoas de 14 anos ou mais de idade, na semana de referência, por sexo e cor ou raça, segundo Grandes Regiões e Unidades da Federação - Brasil - 2022",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Grandes Regiões e Unidades da Federação,Número médio de horas semanais dedicadas aos c...,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Total,Por sexo e cor ou raça,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,Total,NaN,Homem,NaN,Mulher,NaN
6,NaN,NaN,Branca,Preta ou parda,Branca,Preta ou parda,Branca,Preta ou parda
7,Brasil,16.985781,16.54593,17.333142,11.705136,11.74633,20.404106,22.037998
8,Norte,16.187037,15.925246,16.256032,11.588653,11.514646,19.29578,20.534188
9,Rondônia,17.014162,16.727733,17.101523,12.111127,12.54408,20.412887,21.06141


O `.xls` do IBGE mistura **título**, cabeçalho em várias linhas e **notas no rodapé**.  
Cortamos isso e deixamos só: **nomes das colunas + números**.

Tabela 1.1.1 - Número médio de horas semanais dedicadas aos cuidados de pessoas e/ou afazeres domésticos das pessoas de 14 anos ou mais de idade, na semana de referência, por sexo e cor ou raça, segundo Grandes Regiões e Unidades da Federação - Brasil - 2022	

In [15]:
# Lê sem usar a 1ª linha como nome de coluna (senão o título vira "coluna")
bruto = pd.read_excel(path, engine="xlrd", header=None)

# Linhas 0–7: título/cabeçalho  |  8–40: resultados  |  41+: fonte e notas
indicador_1 = bruto.iloc[8:41].copy()
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = indicador_1.reset_index(drop=True)

# horas são número, não texto
for col in indicador_1.columns[1:]:
    indicador_1[col] = pd.to_numeric(indicador_1[col], errors="coerce")

indicador_1


FileNotFoundError: [Errno 2] No such file or directory: 'H:\\python\\3atapa.xlrd'

In [18]:
# Média nacional = linha Brasil (dado do IBGE), não a média das linhas da tabela
media_nacional = indicador_1.loc[indicador_1["uf_regiao"] == "Brasil", "total"].item()
media_nacional

16.985780816563242

In [19]:
media_nacional

16.985780816563242

Qual(is) estado(s) está acima da média nacional de afazeres domésticos?

In [8]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)]

acima_media = estados.loc[
    estados["total"] > media_nacional,
    ["uf_regiao", "total"],
].sort_values("total", ascending=False)

print(f"Média nacional (Brasil): {media_nacional:.2f} horas/semana")
print(f"Estados acima da média: {len(acima_media)}")
acima_media

NameError: name 'indicador_1' is not defined

Quais estados estão abaixo da média?

In [21]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)]

abaixo_media = estados.loc[
    estados["total"] < media_nacional,
    ["uf_regiao", "total"],
].sort_values("total", ascending=False)

print(f"Média nacional (Brasil): {media_nacional:.2f} horas/semana")
print(f"Estados abaixo da média: {len(abaixo_media)}")
abaixo_media

Média nacional (Brasil): 16.99 horas/semana
Estados abaixo da média: 15


,uf_regiao,total
4,Amazonas,16.807290
22,Rio de Janeiro,16.716263
10,Maranhão,16.669614
6,Pará,16.474268
26,Santa Catarina,16.039355
8,Tocantins,15.815781
25,Paraná,15.390122
27,Rio Grande do Sul,15.356709
29,Mato Grosso do Sul,15.309783
31,Goiás,15.186471


Qual é o percentual de diferença no Brasil que mulheres trabalham mais que homens em casa?

In [22]:

brasil = indicador_1.loc[indicador_1["uf_regiao"] == "Brasil"].iloc[0]

homem = (brasil["homem_branca"] + brasil["homem_preta_parda"]) / 2
mulher = (brasil["mulher_branca"] + brasil["mulher_preta_parda"]) / 2

# quanto % a mais as mulheres trabalham em relação aos homens
percentual = (mulher - homem) / homem * 100

print(f"Homens (média das cores):   {homem:.2f} h/semana")
print(f"Mulheres (média das cores): {mulher:.2f} h/semana")
print(f"Mulheres trabalham {percentual:.1f}% a mais que homens em casa, no Brasil.")

# recorte por cor (a tabela traz isso de fato)
perc_branca = (brasil["mulher_branca"] / brasil["homem_branca"] - 1) * 100
perc_preta_parda = (brasil["mulher_preta_parda"] / brasil["homem_preta_parda"] - 1) * 100
print(f"Branca:        mulheres +{perc_branca:.1f}%")
print(f"Preta ou parda: mulheres +{perc_preta_parda:.1f}%")


Homens (média das cores):   11.73 h/semana
Mulheres (média das cores): 21.22 h/semana
Mulheres trabalham 81.0% a mais que homens em casa, no Brasil.
Branca:        mulheres +74.3%
Preta ou parda: mulheres +87.6%


Agora responda:
- Quantas horas mulheres pretas trabalham a mais que mulheres brancas em Minas Gerais?
- Quantas horas mulheres pretas trabalham a mais que homens brancos no Brasil?
- Quantas horas mulheres pretas trabalham a mais que mulheres brancas no Amapá?


In [7]:
# 1. Mulheres pretas/pardas trabalham a mais que mulheres brancas em Minas Gerais

mg = indicador_1.loc[indicador_1["uf_regiao"] == "Minas Gerais"].iloc[0]

diferenca_mg = mg["mulher_preta_parda"] - mg["mulher_branca"]

print(f"Minas Gerais: {diferenca_mg:.2f} horas a mais")

NameError: name 'indicador_1' is not defined

In [ ]:
# 2. Mulheres pretas/pardas trabalham a mais que homens brancos no Brasil

brasil = indicador_1.loc[indicador_1["uf_regiao"] == "Brasil"].iloc[0]

diferenca_brasil = brasil["mulher_preta_parda"] - brasil["homem_branca"]

print(f"Brasil: {diferenca_brasil:.2f} horas a mais")

In [1]:

# 3. Mulheres pretas/pardas trabalham a mais que mulheres brancas no Amapá

amapa = indicador_1.loc[indicador_1["uf_regiao"] == "Amapá"].iloc[0]

diferenca_amapa = amapa["mulher_preta_parda"] - amapa["mulher_branca"]

print(f"Amapá: {diferenca_amapa:.2f} horas")

NameError: name 'indicador_1' is not defined